# Baseline Model — Airline Passenger Satisfaction (S6E10)

> **Pipeline position:** `00_eda` → **00 baseline** → `00_features_experiments` → `01_final_features` → …

| | |
|---|---|
| **Goal** | Establish a solid, untuned LightGBM baseline with cross-validation — the reference score that every later idea must beat. |
| **Metric** | ROC-AUC (probability that a random satisfied passenger gets a higher score than a random unsatisfied one; 0.5 = coin flip, 1.0 = perfect) |
| **Input** | `data/raw/train.csv`, `test.csv`, `sample_submission.csv` |
| **Output** | `submission_lgbm_baseline.csv` (first Kaggle submission) |
| **Runtime** | ~2 min on CPU |
| **Result** | **CV AUC 0.95882** (folds 0.9589 / 0.9577 / 0.9595 / 0.9589 / 0.9592) |

## Table of Contents
1. Setup & Data Loading
2. Preprocessing
3. Cross-Validation
4. Final Model & Submission
5. Results & decisions

> **Glossary:** *Cross-validation (CV)* = split the training data into 5 parts; train on 4, score on the 5th, repeat 5 times. *OOF predictions* = the 5 held-out parts put together: one honest prediction for every training row. *Early stopping* = stop adding trees when the validation score stops improving.

## 1. Setup & Data Loading
Imports, paths (Kaggle or local), and loading. Column names are lower-cased with underscores so they are easy to type and consistent across all notebooks.

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Jump to the project root so relative paths (data/raw, data/processed) work no matter where Jupyter was started. If you clone the repo, change this path (or delete the line and run from the project root).
os.chdir('/Users/deonissx/Documents/Hackathons/kaggle-competitions/predicting_airline_satisfaction_s6e10')

DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e10') if os.path.exists('/kaggle/input') else Path('data/raw')
PROC = Path('data/processed')

print("Imports OK")
print(f"Working directory: {os.getcwd()}")
print(f"Data path: {DATA_DIR}")

Imports OK
Working directory: /Users/deonissx/Documents/Hackathons/kaggle-competitions/predicting_airline_satisfaction_s6e10
Data path: data/raw


In [2]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')

train.columns = train.columns.str.lower().str.replace(' ', '_')
test.columns  = test.columns.str.lower().str.replace(' ', '_')

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")

Train: (699635, 23)
Test:  (299844, 22)


## 2. Preprocessing
Minimal on purpose — features are the subject of the next notebooks.
- **Missing arrival delay** (292 rows, a random pattern found in EDA): fill with the departure delay and add a flag `arrival_delay_missing`.
- **Drop `arrival_delay_in_minutes`:** correlation with departure delay is 0.97, so it is nearly a duplicate.
- **Categoricals** (`gender`, `customer_type`, `type_of_travel`, `class`) → pandas `category`: LightGBM splits on them natively, no one-hot needed.

In [5]:
# NaN flag — before fillna, while the signal is still present
train['arrival_delay_missing'] = train['arrival_delay_in_minutes'].isna().astype(int)
test['arrival_delay_missing']  = test['arrival_delay_in_minutes'].isna().astype(int)

# Only 292 rows have a missing arrival delay, so the fill rule hardly matters; the flag above preserves the information.
# Fix NaN in arrival_delay (292 rows) — MCAR pattern from EDA
train['arrival_delay_in_minutes'] = train['arrival_delay_in_minutes'].fillna(
    train['departure_delay_in_minutes']
)
test['arrival_delay_in_minutes'] = test['arrival_delay_in_minutes'].fillna(
    test['departure_delay_in_minutes']
)

# Target
y = train['satisfaction'].astype(int)

# Features: drop id, target, arrival_delay (duplicate of departure, corr=0.97)
DROP_COLS = ['id', 'satisfaction', 'arrival_delay_in_minutes']

# Categorical features — LightGBM handles them natively
CAT_COLS = ['gender', 'customer_type', 'type_of_travel', 'class']

X = train.drop(columns=DROP_COLS)
X_test = test.drop(columns=['id', 'arrival_delay_in_minutes'])

# Convert categoricals to category dtype — required by LightGBM
for col in CAT_COLS:
    X[col]      = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

print(f"X shape: {X.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\nFeatures ({len(X.columns)}):")
print(list(X.columns))

X shape: (699635, 21)
X_test shape: (299844, 21)

Features (21):
['age', 'flight_distance', 'inflight_wifi_service', 'departure/arrival_time_convenient', 'ease_of_online_booking', 'gate_location', 'food_and_drink', 'online_boarding', 'seat_comfort', 'inflight_entertainment', 'on-board_service', 'leg_room_service', 'baggage_handling', 'checkin_service', 'cleanliness', 'departure_delay_in_minutes', 'gender', 'customer_type', 'type_of_travel', 'class', 'arrival_delay_missing']


## 3. Cross-Validation
Reasonable fixed parameters (no tuning) and a stratified 5-fold split: each fold keeps the same satisfied/unsatisfied share (≈ 44% satisfied).

In [9]:
LGBM_PARAMS = {
    'objective':        'binary',
    'metric':           'auc',
    'learning_rate':    0.05,
    # 127 leaves: fairly flexible trees; lr 0.05 + early stopping keeps training short. Untuned on purpose: this is the REFERENCE every later idea must beat.
    'num_leaves':       127,
    'min_child_samples': 20,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'n_estimators':     1000,
    'early_stopping_rounds': 50,
    'verbose':          -1,
    'random_state':     42,
}

In [ ]:
N_FOLDS = 5
# Fixed 5-fold split (seed 42). Reused in every later notebook, which is what makes all scores comparable and the models stackable.
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_preds  = np.zeros(len(X))          # OOF predictions on train
test_preds = np.zeros(len(X_test))     # averaged predictions on test
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx],  y.iloc[val_idx]

    model = lgb.LGBMClassifier(**LGBM_PARAMS)
    # Early stopping watches the validation fold, which makes fold scores very slightly optimistic. Standard Kaggle practice, identical for every model, so comparisons stay fair.
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        categorical_feature=CAT_COLS,
    )

    val_pred  = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_pred

    test_preds += model.predict_proba(X_test)[:, 1] / N_FOLDS

    score = roc_auc_score(y_val, val_pred)
    fold_scores.append(score)
    print(f"Fold {fold} AUC: {score:.5f}")

# Score on ALL out-of-fold predictions together: the number we compare across the whole project.
cv_score = roc_auc_score(y, oof_preds)
print(f"\nCV AUC (OOF): {cv_score:.5f}")
print(f"Mean fold AUC: {np.mean(fold_scores):.5f} ± {np.std(fold_scores):.5f}")

Fold 1 AUC: 0.95891
Fold 2 AUC: 0.95770
Fold 3 AUC: 0.95948
Fold 4 AUC: 0.95886
Fold 5 AUC: 0.95921

CV AUC (OOF): 0.95882
Mean fold AUC: 0.95883 ± 0.00061


## 4. Final Model & Submission
Test predictions are the **average of the 5 fold models** (no retraining on 100% of the data). Each fold model saw 80% of train; averaging five of them is stable and gives the same-quality prediction for test.

In [11]:
sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
sub['satisfaction'] = test_preds
sub.to_csv('submission_lgbm_baseline.csv', index=False)

print(f"Submission saved. CV AUC: {cv_score:.5f}")
sub.head()

Submission saved. CV AUC: 0.95882


,id,satisfaction
0,699635,0.253934
1,699636,0.853600
2,699637,0.921562
3,699638,0.064541
4,699639,0.042389


## 5. Results & decisions
- **Baseline CV AUC = 0.95882** (std over folds 0.0006). Everything later is measured against this number.
- **Fold 2 is consistently the hardest** (0.9577 here); the same fold is the weakest for every later model too, so it reflects *that split*, not model quality. This is why we compare models fold by fold on the identical split.
- **Differences below ≈ 0.0003 are noise** (fold-to-fold spread and public-LB noise), so a new idea must clear a higher bar before we keep it.
- **Next:** `00_features_experiments` tests feature ideas one at a time against this baseline.